# 00 — Menyiapkan Google Colab untuk Training

**Tujuan:** menyiapkan runtime Colab, mengambil kode proyek, memasang paket,
menyambungkan Google Drive, dan membuat split data — tanpa menjalankan training
berat. Setelah notebook ini selesai, `data/processed/train.csv`, `val.csv`, dan
`test.csv` siap dipakai.

**Prasyarat:** akun Google, repo sudah di-push ke GitHub **atau** zip proyek di
Drive, dan `re_dataset.csv` sudah diunggah ke `MyDrive/cyberbully/data/`.

**Estimasi waktu:** 10–15 menit (unduhan paket dan model tokenizer bisa lama
tergantung jaringan).

**Kenapa setup dipisah dari training?** Runtime Colab bisa mati kapan saja.
Dengan memisahkan setup, kita bisa mengulang setup dengan cepat lalu melanjutkan
training dari checkpoint, tanpa mengulang eksperimen yang sudah jadi.

## Glosarium

- **Runtime**: komputer virtual milik Google tempat notebook berjalan. Kalau tab
  ditutup atau idle terlalu lama, runtime dihentikan dan semua file di `/content`
  hilang.
- **GPU (T4)**: prosesor grafis untuk mempercepat operasi matriks. Training
  transformer jauh lebih cepat di GPU daripada CPU.
- **Google Drive**: penyimpanan permanen milik akun Google. Berbeda dari
  `/content`, isinya tidak hilang saat runtime mati.
- **`git clone`**: menyalin repo dari GitHub ke `/content`.
- **`pip install -e`**: memasang paket dalam mode *editable*, sehingga perubahan
  kode langsung terbaca tanpa install ulang.
- **Split**: pembagian data menjadi train (belajar), validation (memilih
  hyperparameter/threshold), dan test (laporan akhir, hanya sekali).

## 1. Cek runtime

Sel pertama hanya memeriksa identitas runtime. Kami perlu tahu apakah kode
berjalan di Colab atau di Mac, karena perintah mount Drive hanya ada di Colab.
Tidak ada data yang diubah di langkah ini.

In [ ]:
import platform
import sys

IN_COLAB = "google.colab" in sys.modules
print("Python :", platform.python_version())
print("Sistem :", platform.platform())
print("Di Colab:", IN_COLAB)

## 2. Cek GPU

Training IndoBERT di CPU bisa memakan waktu berkali-kali lipat. Di Colab,
pastikan GPU aktif: **Runtime → Change runtime type → T4 GPU**. Jika
`torch.cuda.is_available()` bernilai `False`, hentikan di sini dan ubah runtime
type; menjalankan training tetap bisa, tetapi sangat lambat.

In [ ]:
import torch

print("torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
else:
    print("GPU tidak aktif. Di Mac, MPS adalah pengganti yang wajar.")

## 3. Ambil kode proyek

Ada dua jalur. **Opsi A (disarankan):** isi `REPO_URL` dengan URL repo GitHub
Anda, lalu sel ini menjalankan `git clone`. **Opsi B:** kalau belum memakai
GitHub, unggah zip proyek ke Drive, ekstrak ke `/content`, dan biarkan
`REPO_URL` kosong. Sel ini sengaja tidak menghapus folder yang sudah ada agar
pekerjaan sebelumnya tidak hilang.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = ""  # contoh: "https://github.com/username/cyberbully.git"
PROJECT_DIR = Path("/content/CyberBully")

if REPO_URL and not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
    print("Repo di-clone ke", PROJECT_DIR)
elif REPO_URL:
    print("Folder sudah ada; lewati clone. Jalankan `git pull` manual bila perlu.")
elif PROJECT_DIR.exists():
    print("Memakai folder yang sudah ada:", PROJECT_DIR)
else:
    print("REPO_URL kosong dan folder belum ada.")
    print("Unggah zip ke Drive, lalu ekstrak ke /content sebelum melanjutkan.")

## 4. Pasang paket proyek

`pip install -e ".[dev]"` memasang paket `cyberbully` beserta dependensi.
Tanda `-e` berarti *editable*: kode di `src/` dibaca langsung, bukan disalin.
Di Colab, `torch` sudah tersedia sehingga tidak diunduh ulang karena versinya
memenuhi syarat. Di Mac, sel ini dilewati supaya environment `ml-lab` tidak
berubah tanpa sengaja.

In [ ]:
import os

if IN_COLAB:
    os.chdir(PROJECT_DIR)
    get_ipython().run_line_magic("pip", 'install -q -e ".[dev]"')
    print("Paket terpasang.")
else:
    print("Bukan Colab: install dilewati. Pakai environment `ml-lab`.")

## 5. Mount Google Drive

Drive dipakai untuk menyimpan dataset dan checkpoint agar tidak hilang saat
runtime berhenti. Proses mount akan meminta izin akses akun Google. Di luar
Colab, langkah ini dilewati dengan aman.

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive/cyberbully")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive terpasang di", DRIVE_DIR)
else:
    print("Bukan Colab: mount Drive dilewati.")

## 6. Siapkan dataset

Dataset **tidak** disimpan di repo karena lisensi CC BY-NC-SA 4.0 dan aturan
privasi. Letakkan `re_dataset.csv` di `MyDrive/cyberbully/data/`, lalu sel ini
menyalinnya ke `data/raw/` di runtime. Membaca data dari Drive saat training
lambat, jadi salinan lokal ini penting.

In [ ]:
import shutil

repo = PROJECT_DIR if PROJECT_DIR.exists() else Path.cwd().resolve()
raw_target = repo / "data" / "raw" / "re_dataset.csv"
raw_source = DRIVE_DIR / "data" / "re_dataset.csv"

if raw_target.exists():
    print("Dataset sudah ada:", raw_target)
elif raw_source.exists():
    raw_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_source, raw_target)
    print("Dataset disalin ke", raw_target)
else:
    print("Dataset tidak ditemukan.")
    print("Letakkan re_dataset.csv di", raw_source, "lalu ulangi sel ini.")

## 7. Jalankan pipeline data

Perintah di bawah memuat CSV, memperbaiki emoji yang ter-escape, menggabungkan
label `HS` dan `Abusive` menjadi `bully`, membuang duplikat, lalu membagi data
80/10/10 dengan seed 42. Deduplikasi dilakukan **sebelum** split supaya teks
yang sama tidak muncul di train dan test (data leakage).

In [ ]:
if raw_target.exists():
    import subprocess
    import sys

    cmd = [
        sys.executable, "-m", "cyberbully.data",
        "--input", str(raw_target),
        "--text-col", "Tweet",
        "--label-col", "HS", "Abusive",
        "--positive", "1",
        "--label-mode", "any",
        "--keep-cols", "HS", "Abusive",
    ]
    result = subprocess.run(cmd, cwd=repo, text=True)
    print("exit code:", result.returncode)
else:
    print("Dilewati: dataset belum tersedia.")

## 8. Periksa hasil split

Angka yang diharapkan untuk dataset ini: sekitar 12.997 baris bersih, dengan
train 10.397, validation 1.300, dan test 1.300. Kalau angkanya jauh berbeda,
jangan lanjut training — periksa dulu file masukan dan pesan peringatan encoding.

In [ ]:
import pandas as pd

processed = repo / "data" / "processed"
if (processed / "train.csv").exists():
    for name in ("train", "val", "test"):
        part = pd.read_csv(processed / f"{name}.csv")
        counts = part["label"].value_counts().sort_index().to_dict()
        print(f"{name:5s} {len(part):5d} baris | label={counts}")
else:
    print("Split belum ada; jalankan sel pipeline data di atas.")

## Ringkasan

Kalau semua sel di atas berhasil, runtime sudah punya kode, paket, dataset, dan
split yang siap dilatih. Langkah berikutnya ada di
`06_colab_training.ipynb`: smoke test model kecil, lalu training IndoBERT
dengan checkpoint ke Drive.

Yang **belum** dilakukan di sini: training, pemilihan threshold, dan evaluasi
test set. Test set sengaja tidak disentuh sampai laporan akhir.